## Read the data

In [84]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"
EXPERIMENT= "paper2_core_clustering"

# 1. Read data (Now it is the same data)

In [85]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/core_clustering_paper/per_length_per_kfold_split/' # work laptop
APPROACH = "DECOY_PATHS"

In [86]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
# per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
# per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

target_class = "class1"


DataParams = {
"class1": {
    "ALLELES": sorted([ 'HLA-A*01:01','HLA-A*03:01']),
    "TARGET_LENGTHS":   [8, 9, 11, 12,13,14,15],
},
    
"class2": {
    "ALLELES":sorted(['HLA-DRB1*01:01']),
    "TARGET_LENGTHS":    [12, 13,14, 15, 16,17, 18],

    
    
}

}

ALLELES = DataParams[target_class]["ALLELES"]
TARGET_LENGTHS = DataParams[target_class]["TARGET_LENGTHS"]

selected_alleles = ALLELES
ALLELES




['HLA-A*01:01', 'HLA-A*03:01']

## 2.0 Merge alleles together

In [87]:
# combination_size = len(ALLELES)
# ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
# # ALLELES = sorted([ 'HLA-A*03:01'])
# ALLELES = sorted([   'HLA-DRB1*01:01',
#                     # 'HLA-DPA1*01:03/DPB1*04:01'
#                       ])
# ALLELES = sorted(['HLA-DPA1*01:03-DPB1*04:01'])
#
# ALLELES = sorted(['HLA-DRB1*01:01', 'HLA-DPA1*01:03-DPB1*04:01'])
#

combination_size = 2
TOTAL_SPLITS = 1
# TARGET_LENGTHS= [8, 9, 10, 11, 12]
RUNS = 30
USE_WEIGHTS = True

# SELECTED_RUN = 0
# SELECTED_RUN = {0,1,2,3,4,5, 6, 7, 8, 9}
SELECTED_RUN = {0,1}

# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
models_to_read = ['v']

## Merge train data together (multiple alleles to single set of peptides)

In [88]:
from collections import defaultdict
global_selected_alleles = ALLELES # Data alleles we selected (but the number will be reduced by acceptable alleles)

selected_alleles = global_selected_alleles


per_allele_representations_train = dict()
per_allele_representations_test = dict()
per_name_representation_train = dict()
per_name_representation_test = dict()

def handle_intersections(per_allele_per_kfold_per_length_peptides):
    per_allele_result_peptide_sets =  defaultdict(set)
    per_allele_per_length_peptide_sets =  defaultdict(set)
    PER_LENGTH_SIZE = {target_length: min([len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])) for allele_name in selected_alleles]) for target_length in TARGET_LENGTHS}
    print("Min unique sizes", PER_LENGTH_SIZE)
    for target_length in TARGET_LENGTHS:
        print(f"Length {target_length}")
        for allele_name in selected_alleles:
            if target_length in per_allele_per_kfold_per_length_peptides[allele_name][0]:
                print(f"initial length for  allele { allele_name} TL {target_length} is {len(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])}")
                print(f"unique length for  allele { allele_name} TL {target_length} is {len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length]))}")
                unique_peptides = np.unique(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])
                if "imm" not in allele_name: #do not subsambple for imm
                    peptides = set(np.random.choice(unique_peptides, size=PER_LENGTH_SIZE[target_length], replace=False))
                else:
                    peptides = set(unique_peptides)
                print(f"Selected/Subsampled {len(peptides)} peptides")
                per_allele_per_length_peptide_sets[allele_name] = peptides
        # get all intersections
        intersections = list()
        for i in range(len(selected_alleles)):
            for j in range(i + 1, len(selected_alleles)):
                set1 = per_allele_per_length_peptide_sets[selected_alleles[i]]
                set2 = per_allele_per_length_peptide_sets[selected_alleles[j]]
                intersect = set1.intersection(set2)
                print(f"{selected_alleles[i]} intersect {selected_alleles[j]} - {len(intersect)}")
                intersections.append(intersect)
        # delete all intersections
        for allele_name, peptide_set in per_allele_per_length_peptide_sets.items():
            print(f"Allele {allele_name}: deleted", end=" ")
            initial_length = len(peptide_set)
            for intersect in intersections:
                peptide_set = peptide_set.difference(intersect)
            result_length = len(peptide_set)
            print(f" {initial_length - result_length} of intersected peptides")
            per_allele_result_peptide_sets[allele_name] = per_allele_result_peptide_sets[allele_name].union(peptide_set)
    return per_allele_result_peptide_sets

print("Train")
per_allele_peptide_train_sets = handle_intersections(per_allele_per_kfold_per_length_binders_train)
print("Test")
per_allele_peptide_test_sets = handle_intersections(per_allele_per_kfold_per_length_binders_test)



train_peptides_b = set(set().union(*per_allele_peptide_train_sets.values()))

test_peptides_b = set(set().union(*per_allele_peptide_test_sets.values()))

print(f"Train {len(train_peptides_b)}, Test {len(test_peptides_b)}")



data_df = pd.DataFrame(index=list(train_peptides_b))
for allele_name in selected_alleles:
    data_df.loc[list(per_allele_peptide_train_sets[allele_name]), 'allele'] = allele_name



Train
Min unique sizes {8: 96, 9: 4206, 11: 724, 12: 214, 13: 106, 14: 37, 15: 16}
Length 8
initial length for  allele HLA-A*01:01 TL 8 is 798
unique length for  allele HLA-A*01:01 TL 8 is 763
Selected/Subsampled 96 peptides
initial length for  allele HLA-A*03:01 TL 8 is 116
unique length for  allele HLA-A*03:01 TL 8 is 96
Selected/Subsampled 96 peptides
HLA-A*01:01 intersect HLA-A*03:01 - 0
Allele HLA-A*01:01: deleted  0 of intersected peptides
Allele HLA-A*03:01: deleted  0 of intersected peptides
Length 9
initial length for  allele HLA-A*01:01 TL 9 is 5388
unique length for  allele HLA-A*01:01 TL 9 is 4721
Selected/Subsampled 4206 peptides
initial length for  allele HLA-A*03:01 TL 9 is 5011
unique length for  allele HLA-A*03:01 TL 9 is 4206
Selected/Subsampled 4206 peptides
HLA-A*01:01 intersect HLA-A*03:01 - 53
Allele HLA-A*01:01: deleted  53 of intersected peptides
Allele HLA-A*03:01: deleted  53 of intersected peptides
Length 11
initial length for  allele HLA-A*01:01 TL 11 is 170

In [89]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets

per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

HLA-A*01:01+HLA-A*03:01
Length 8: HLA-A*01:01 - 798 HLA-A*03:01 - 116 
Length 9: HLA-A*01:01 - 5388 HLA-A*03:01 - 5011 
Length 10: HLA-A*01:01 - 2776 HLA-A*03:01 - 2265 
Length 11: HLA-A*01:01 - 1700 HLA-A*03:01 - 880 
Length 12: HLA-A*01:01 - 1328 HLA-A*03:01 - 252 
Length 13: HLA-A*01:01 - 965 HLA-A*03:01 - 127 
Length 14: HLA-A*01:01 - 602 HLA-A*03:01 - 37 
Length 15: HLA-A*01:01 - 483 HLA-A*03:01 - 16 
Length 16: HLA-A*01:01 - 348 HLA-A*03:01 - 13 
Length 17: HLA-A*01:01 - 210 HLA-A*03:01 - 5 
Length 18: HLA-A*01:01 - 128 HLA-A*03:01 - 5 


In [90]:
selected_alleles

['HLA-A*01:01', 'HLA-A*03:01']

In [91]:
data_df.allele.value_counts()

HLA-A*01:01    5344
HLA-A*03:01    5344
Name: allele, dtype: int64

# Read models

In [92]:
MODELS_DESCRIPTIONS = {

       "class_I_simple_clustering": {
        "path": "experiment_results/paper2_core_clustering/sorted_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size": 2,
        "alleles": [
                # 'HLA-DRB1*01:01'
                    'HLA-A*01:01',
                    'HLA-A*03:01'
                    ],
        "target_lengths": [8, 9,10, 11, 12,13, 14],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },

    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [93]:
selected_models = ["class_I_simple_clustering"]


In [94]:
MODELS_DESCRIPTIONS

{'class_I_simple_clustering': {'path': 'experiment_results/paper2_core_clustering/sorted_models/',
  'states': 15,
  'params_string': '60-min_iters-4-aaas',
  'combination_size': 2,
  'alleles': ['HLA-A*01:01', 'HLA-A*03:01'],
  'target_lengths': [8, 9, 10, 11, 12, 13, 14],
  'core_start_state': [15, 16],
  'core_end_state': [17, 18],
  'need_to_delete': [],
  'skip_states': []}}

In [95]:
MODELS_DESCRIPTIONS

{'class_I_simple_clustering': {'path': 'experiment_results/paper2_core_clustering/sorted_models/',
  'states': 15,
  'params_string': '60-min_iters-4-aaas',
  'combination_size': 2,
  'alleles': ['HLA-A*01:01', 'HLA-A*03:01'],
  'target_lengths': [8, 9, 10, 11, 12, 13, 14],
  'core_start_state': [15, 16],
  'core_end_state': [17, 18],
  'need_to_delete': [],
  'skip_states': []}}

In [96]:
from hmm_logic_methods import load_model

In [97]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_binding_model = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_binding_model[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [98]:
model_alleles = model_description['alleles']
MIXED_ALLELES = list()

model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
for combination in itertools.combinations(sorted(model_alleles), combination_size):
    allele_name = "+".join(combination)
    # allele_name = f"{allele_name1}+{allele_name2}"
    MIXED_ALLELES.append(allele_name)

In [99]:
model_description['alleles']

['HLA-A*01:01', 'HLA-A*03:01']

In [100]:
per_name_binding_model[model_name].states

[{
     "class" : "State",
     "distribution" : {
         "class" : "Distribution",
         "dtype" : "str",
         "name" : "DiscreteDistribution",
         "parameters" : [
             {
                 "A" : 0.002614819660124338,
                 "C" : 0.0026250230566174605,
                 "D" : 0.03929791697757267,
                 "E" : 0.025096242914105487,
                 "F" : 0.07165927351484504,
                 "G" : 0.048204889768293596,
                 "H" : 0.002614819660124338,
                 "I" : 0.15447232248701365,
                 "K" : 0.003592120759025437,
                 "L" : 0.002614819660124338,
                 "M" : 0.002614819660124338,
                 "N" : 0.002614819660124371,
                 "P" : 0.009281486326791004,
                 "Q" : 0.002614819660124339,
                 "R" : 0.002614887568289216,
                 "S" : 0.002614819660124338,
                 "T" : 0.6000084960123222,
                 "V" : 0.002614819660124338,

In [101]:
OLD_ALLELES

['HLA-A*01:01', 'HLA-A*03:01']

In [102]:
model_alleles

{'HLA-A*01:01', 'HLA-A*03:01'}

In [103]:
MODELS_DESCRIPTIONS[model]

{'path': 'experiment_results/paper2_core_clustering/sorted_models/',
 'states': 15,
 'params_string': '60-min_iters-4-aaas',
 'combination_size': 2,
 'alleles': ['HLA-A*01:01', 'HLA-A*03:01'],
 'target_lengths': [8, 9, 10, 11, 12, 13, 14],
 'core_start_state': [15, 16],
 'core_end_state': [17, 18],
 'need_to_delete': [],
 'skip_states': []}

In [104]:
MODELS_DESCRIPTIONS[model]

{'path': 'experiment_results/paper2_core_clustering/sorted_models/',
 'states': 15,
 'params_string': '60-min_iters-4-aaas',
 'combination_size': 2,
 'alleles': ['HLA-A*01:01', 'HLA-A*03:01'],
 'target_lengths': [8, 9, 10, 11, 12, 13, 14],
 'core_start_state': [15, 16],
 'core_end_state': [17, 18],
 'need_to_delete': [],
 'skip_states': []}

## Mark important states in model(will be used for core call)

In [105]:
import logomaker
from collections import defaultdict
def probs_to_info_matrix(distribution):
    amino_acids = list(distribution.parameters[0].keys())
    dummy_df = pd.DataFrame(columns = sorted(amino_acids), index=[0],  dtype=float)
    for i, (acid, prob) in enumerate(distribution.parameters[0].items()):
        dummy_df.loc[0, acid] = prob
    dummy_df.index.name = "pos"
    info_matrix = logomaker.transform_matrix(dummy_df, from_type='probability', to_type='information')
    return info_matrix

per_model_total_df = dict()

for model_key, model in per_name_binding_model.items():
    per_state_information = dict()
    per_state_distribution = dict()
    per_level_information = defaultdict(float)

    for state in model.states:
        if state not in [model.start, model.end]:
            info_matrix = probs_to_info_matrix(state.distribution)
            per_state_distribution[state.name] = state.distribution.parameters[0]
            per_state_information[state.name] = sum(info_matrix.loc[0])
            # per_level_information[name_to_level[state.name]] += per_state_information[state.name]
    info_df = pd.DataFrame(per_state_information.items(), columns=["state", "info"])
    distr_df = pd.DataFrame(per_state_distribution.values())
    total_df = pd.concat([info_df, distr_df], axis=1)
    per_model_total_df[model_key] = total_df

In [106]:
per_model_total_df[model_key]

,state,info,A,C,D,E,F,G,H,I,...,M,N,P,Q,R,S,T,V,W,Y
0,s000,2.213300,0.002615,0.002625,0.039298,0.025096,0.071659,0.048205,0.002615,0.154472,...,0.002615,0.002615,0.009281,0.002615,0.002615,0.002615,0.600008,0.002615,0.019614,0.002615
1,s001,2.283279,0.142047,0.002183,0.002183,0.024747,0.002183,0.002183,0.002183,0.002183,...,0.002183,0.002211,0.052942,0.002183,0.002183,0.002183,0.002183,0.241671,0.002188,0.002183
2,s002,2.662779,0.003258,0.003258,0.003258,0.003258,0.003258,0.003258,0.003258,0.003258,...,0.181984,0.003258,0.003258,0.014037,0.003315,0.666046,0.003258,0.003258,0.003258,0.085751
3,s003,1.124952,0.109347,0.001913,0.009977,0.324471,0.057005,0.001400,0.053276,0.016481,...,0.053126,0.051914,0.090333,0.036323,0.001729,0.023566,0.005371,0.001401,0.015759,0.141860
4,s004,1.401400,0.007520,0.001625,0.001618,0.153121,0.001618,0.269533,0.002427,0.137951,...,0.013177,0.001618,0.001618,0.082654,0.015264,0.001618,0.159851,0.124281,0.009816,0.001618
5,s005,2.214569,0.001319,0.001319,0.568288,0.001319,0.042162,0.001530,0.001319,0.001319,...,0.001319,0.044619,0.001319,0.001319,0.037589,0.094880,0.001319,0.001319,0.001319,0.001745
6,s006,1.905452,0.001658,0.011026,0.107170,0.001651,0.001651,0.001651,0.001651,0.059855,...,0.001651,0.001651,0.001651,0.001651,0.077403,0.001651,0.157312,0.103094,0.001651,0.001651
7,s007,1.723550,0.090283,0.001642,0.001435,0.001969,0.001435,0.001435,0.084570,0.001435,...,0.010007,0.001435,0.294563,0.077810,0.001435,0.158795,0.001435,0.001435,0.001456,0.001435
8,s008,1.770336,0.002432,0.002432,0.002432,0.002432,0.211447,0.321868,0.002432,0.002432,...,0.002432,0.236356,0.052484,0.002432,0.002432,0.002432,0.002432,0.002432,0.055287,0.088511
9,s009,1.332844,0.007573,0.025899,0.099463,0.002007,0.005021,0.150836,0.036003,0.071929,...,0.034051,0.002007,0.002007,0.002007,0.114650,0.002027,0.337488,0.002009,0.008182,0.092825


In [107]:
filtered_dfs = list()
per_model_important_states = dict()
per_model_skip_states = dict()

if models_to_read == ['bw']:
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df[total_df['info'] >= 1]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)

if models_to_read == ['v']:
    per_model_anchors = dict()
    per_name_models = per_name_binding_model

    core_start_states= MODELS_DESCRIPTIONS[selected_models[0]]['core_start_state']
    core_end_states = MODELS_DESCRIPTIONS[selected_models[0]]['core_end_state']
    skip_states =  MODELS_DESCRIPTIONS[selected_models[0]]['skip_states']

    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[core_start_states + core_end_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[skip_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_skip_states[model_key] = list(total_df['state'].values)
        





## Calculate core guesses

In [108]:
# prepare peptides for viterbi calculation (expected to be unique)
allele_peptides = data_df.index.values


len(allele_peptides)

10688

In [109]:
data_df.index.str.len().value_counts()

9     8306
11    1446
12     426
13     212
8      192
14      74
15      32
dtype: int64

In [110]:
TEST_PDB_CORES = False
if TEST_PDB_CORES:
    df=pd.read_csv('test/binding_cores', sep='\t')
    df= df[~df['Antigen'].str.contains('X')]
    df = df[df['Allele'] == 'DRB1*01:01']

    allele_peptides = df.Antigen.values
    true_labels=df['Core (PDB)'].values
    netmhcpan = df['Predicted core.5'].values

In [111]:
from hmm_logic_methods import calculate_model_core_guesses
selected_peptides = [
    # "APEEILAEKSFNTVPY", 'EALTFSREHGPGVAEPE',
    #                  'YYSLLMPILTLTRAL',
    #                      'SYLTNMFATWSP',
    #                  'NPLIRHENRMVLAST',
    #                      'KPPKPVSKMRMATPLLM',
                         ]
total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states, per_model_skip_states=per_model_skip_states,
                                    selected_peptides=selected_peptides
                                   )

Model 0_run-8-9-10-11-12-13-14-15-st-viterbi-alg-60-min_iters-4-aaas_model-HLA_A_01_01+HLA_A_03_01-0
Get viterbi paths for each peptide...


100%|██████████| 10688/10688 [00:00<00:00, 48345.78it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 10688 


100%|██████████| 10688/10688 [00:00<00:00, 37605.59it/s]


Model 1_run-8-9-10-11-12-13-14-15-st-viterbi-alg-60-min_iters-4-aaas_model-HLA_A_01_01+HLA_A_03_01-0
Get viterbi paths for each peptide...


100%|██████████| 10688/10688 [00:00<00:00, 51637.90it/s]


... (110 outputs left out)

Model 29_run-8-9-10-11-12-13-14-15-st-viterbi-alg-60-min_iters-4-aaas_model-HLA_A_01_01+HLA_A_03_01-0
Get viterbi paths for each peptide...


100%|██████████| 10688/10688 [00:00<00:00, 49572.29it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 10688 


100%|██████████| 10688/10688 [00:00<00:00, 37791.17it/s]
C:\Projects\MHC_predictor\hmm_logic_methods.py:1762: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'core_start_{i}'] =  viterbi_paths['simplified_path'].apply(lambda x: np.where(np.array(x) > 0)[0]).apply(lambda x: x[0] if len(x) > 0 else 0)
C:\Projects\MHC_predictor\hmm_logic_methods.py:1763: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'core_end_{i}'] =  viterbi_paths['simplified_path'].apply(lambda x: np.where(np.array(x) > 0)[0]).apply(lambda x: x[

In [112]:
data_df = data_df.join(total_df)


In [113]:
if TEST_PDB_CORES:
    models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
    models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
    core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
    core_ends = models_predictions_end.mode(axis=1)[0].astype(int)
    
    total_df['core_start'] = core_starts
    total_df['core_end'] = core_ends

    total_df['core_peptide'] = total_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
    total_df['core_len'] = total_df['core_peptide'].str.len()
    total_df['peptide_len'] = total_df.index.str.len()
    total_df['core_peptide'] = total_df['core_peptide']
    total_df = total_df[~total_df.index.duplicated(keep='first')]


    total_df.to_csv('test/predicted_cores_cores.csv',sep=";", index=True)

In [114]:
total_df.index.str.len().value_counts()

9     8306
11    1446
12     426
13     212
8      192
14      74
15      32
dtype: int64

## Set core based on most popular value

In [115]:
from scipy import stats

models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
model_predictions_cores = total_df[[f"core_{i}" for i in range(RUNS)]]
core_consensuses = model_predictions_cores.apply(lambda x: 
                stats.mode(np.vstack(x.values))[0][0], axis=1)


C:\Users\denkl\AppData\Local\Temp\ipykernel_33492\2287683951.py:7: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  stats.mode(np.vstack(x.values))[0][0], axis=1)


In [116]:

core_consensuses

YTDMESEDY            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
RLFERSRIK            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
FVELPDNHSSLIY    [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
SLPPVLKAK            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
KTKPPALTK            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
                                       ...                        
RLRAQPVTK            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
VADAIAQKY            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
YLESHGLAY            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
ITQGTPLKY            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
PIDGNFFTY            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Length: 10688, dtype: object

In [117]:
len([item for item in core_consensuses if 0 in item])

0

In [118]:
core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
core_ends = models_predictions_end.mode(axis=1)[0].astype(int)


In [119]:
final_df = total_df

In [120]:
selected_cluster_column = 'min_entropy_cluster'
#selected_cluster_column = 'max_mean_prob_cluster'
# final_df['core_start'] = 0
# final_df['core_end'] = 0
# for cluster in cluster_descriptions.index:
#     index_peptides = final_df[final_df[selected_cluster_column] == cluster].index
#     start_values = cluster_to_start_columns[cluster]
#     end_values = cluster_to_end_columns[cluster]
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mode(axis=1)[0].astype(int)

#     #final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mode(axis=1)[0].astype(int)

final_df['core_start'] = core_starts
final_df['core_end'] = core_ends

#final_df['core_peptide'] = final_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
final_df['core_peptide'] = final_df.apply(lambda row: "".join(np.array(list(row.name))[core_consensuses.loc[row.name].astype(bool)]), axis=1)

final_df['core_len'] = final_df['core_peptide'].str.len()

In [121]:
final_df['core_len'].value_counts()

9     8306
11    1446
12     426
13     212
8      192
14      74
15      32
Name: core_len, dtype: int64

In [122]:
data_df['core_start'] = final_df['core_start']
data_df['core_end'] = final_df['core_end']
data_df['core_len'] = final_df['core_len']
data_df['peptide_len'] = data_df.index.str.len()
data_df['core_peptide'] = final_df['core_peptide']
data_df = data_df[~data_df.index.duplicated(keep='first')]

C:\Users\denkl\AppData\Local\Temp\ipykernel_33492\1549749149.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_df['core_start'] = final_df['core_start']
C:\Users\denkl\AppData\Local\Temp\ipykernel_33492\1549749149.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_df['core_end'] = final_df['core_end']
C:\Users\denkl\AppData\Local\Temp\ipykernel_33492\1549749149.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

In [123]:
data_df['core_start'].value_counts()

0    10688
Name: core_start, dtype: int64

# Save core data to files

In [130]:
import os

PATH_TO_SAVE_RESULTS = f"{'/'.join(name_to_model_root[model_name].split('/')[0:2])}/peptide_cores/"
if not os.path.exists(PATH_TO_SAVE_RESULTS):
    os.makedirs(PATH_TO_SAVE_RESULTS)
data_df.to_csv("{}extractd_cores.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=True)
data_df.peptide.to_csv("{}original_peptides.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=False)

In [125]:
"{}extractd_cores.csv".format(PATH_TO_SAVE_RESULTS,)

'experiment_results/paper2_core_clustering/sorted_models//peptide_cores/extractd_cores.csv'

# Group models based on predictions

We got all the start predictions from all models. Now we want to group these models. (since we expect that models have disagrement)

We want to cluster columns based on how similar are predictions for the whole training data



In [82]:
models_predictions = total_df[[f"core_start_{i}" for i in range(RUNS)]]

In [ ]:
from sklearn.cluster import FeatureAgglomeration
distance_threshold = 150

agglo = FeatureAgglomeration(n_clusters=None, metric='euclidean', linkage='ward', pooling_func=np.mean, distance_threshold=distance_threshold)
result_start_df = agglo.fit_transform(models_predictions)

In [78]:
clustered_features = pd.DataFrame({"cluster": agglo.labels_})

clustered_features['model'] = models_predictions.columns


unique_labels_counts = clustered_features.cluster.value_counts().sort_values(ascending=False)
unique_labels_counts

1    15
2    10
3     6
0     4
Name: cluster, dtype: int64

In [79]:
cluster_descriptions = clustered_features.groupby('cluster').agg(lambda x: " ".join(x.values))
cluster_descriptions['len'] = unique_labels_counts
cluster_descriptions.sort_values(by='len', ascending=False)

,model,len
cluster,,
1,core_start_1 core_start_3 core_start_10 core_s...,15
2,core_start_2 core_start_5 core_start_6 core_st...,10
3,core_start_9 core_start_11 core_start_23 core_...,6
0,core_start_0 core_start_4 core_start_17 core_s...,4


In [80]:
cluster_to_start_columns = {i : total_df[cluster_descriptions.loc[i]['model'].split(' ')] for i in cluster_descriptions.index}

In [81]:
model_to_cluster = dict(clustered_features['cluster'])

## Add mean probability for cluster for each peptide

In [34]:
from collections import defaultdict
cluster_to_prob_column_names = defaultdict(list)
cluster_to_end_column_names = defaultdict(list)

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = 0
    
# calculate mean for predictions for given cluster of models to 
for i in range(RUNS):
    total_df[f'cluster_prob_{model_to_cluster[i]}'] += total_df[f'prob_{i}']
    cluster_to_prob_column_names[model_to_cluster[i]].append(f'prob_{i}')
    cluster_to_end_column_names[model_to_cluster[i]].append(f'core_end_{i}')

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = total_df[f'cluster_prob_{cluster}'] / cluster_descriptions.loc[cluster]['len']
    
total_df['max_mean_prob_cluster'] =   total_df[[f'cluster_prob_{cluster}' for cluster in cluster_descriptions.index]].idxmax(axis=1).str.split('_').map(lambda x : int(x[-1]))

cluster_to_prob_columns = {i : total_df[cluster_to_prob_column_names[i]] for i in cluster_to_prob_column_names}
cluster_to_end_columns = {i : total_df[cluster_to_end_column_names[i]] for i in cluster_to_end_column_names}


0
1
2
3
0
1
2
3


In [35]:
cluster_to_start_columns[1].shape

(5640, 15)

## Add entropy value for cluster for each column

In [36]:
def mean_predictive_entropy(predictions):
    entropy = np.mean(-np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1])
    return entropy

def predictive_entropy(predictions):
    entropy = -np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1]
    return entropy

print(mean_predictive_entropy(cluster_to_prob_columns[0]))
print(mean_predictive_entropy(cluster_to_prob_columns[1]))
#print(mean_predictive_entropy(cluster_to_prob_columns[2]))
#print(mean_predictive_entropy(cluster_to_prob_columns[3]))

cluster_to_start_entropy_df = {i: predictive_entropy(cluster_to_prob_columns[i]) for i in cluster_to_prob_columns.keys()}

0.15289490292766392
0.15267749732179514


In [37]:
df_concat = pd.concat(cluster_to_start_entropy_df.values(), axis=1)
min_entropy_clusters = df_concat.idxmin(axis=1)
df_concat.columns = [f'cluster_entropy_{cluster}' for cluster in df_concat.columns ]
df_concat['min_entropy_cluster'] = min_entropy_clusters

In [38]:
final_df = pd.concat([total_df, df_concat], axis=1)